# Working with xarray

Open bands lazily from S3, clip to the AOI, mask clouds with the scene classification (SCL), and plot an NDVI time series. You need `jupyterlab-jstex[s3]`, rioxarray, xarray, pandas, matplotlib (dask for larger areas).

In [ ]:
import jstex

jstex.whoami()  # 'anonymous'? Sign in in the widget below or run jstex.login()

In [ ]:
ex = jstex.Explorer()
ex

Choose a Sentinel-2 L2A item in the widget (it becomes `ex.selected_item`), or run the next cell to search from code.

In [ ]:
AOI = [10.0, 45.3, 10.4, 45.6]  # lon/lat bbox (Lake Garda, Italy)
ex.query = {
    **ex.query,
    "collections": ["sentinel-2-l2a"],
    "datetime": {"from": "2024-07-01T00:00:00Z", "to": "2024-07-31T23:59:59Z"},
    "aois": [
        {
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [AOI[0], AOI[1]],
                        [AOI[2], AOI[1]],
                        [AOI[2], AOI[3]],
                        [AOI[0], AOI[3]],
                        [AOI[0], AOI[1]],
                    ]
                ],
            },
            "selected": True,
        }
    ],
    "filters": [{"field": "eo:cloud_cover", "op": "<=", "value": 30}],
}
ex.search(wait=True)
item = ex.selected_item or next(iter(ex.results))
item.id, sorted(item.assets)

In [ ]:
def band(item, common_name):
    """The finest-resolution single-band asset of a band, by its common name (e.g. 'red', 'nir')."""
    found = []
    for key, asset in item.assets.items():
        bands = (
            asset.extra_fields.get("eo:bands") or asset.extra_fields.get("bands") or []
        )
        if any(
            common_name in (b.get("common_name"), b.get("eo:common_name"))
            for b in bands
        ):
            # finest gsd first; single-band assets before composites such as TCI
            found.append((asset.extra_fields.get("gsd") or 999, len(bands), key, asset))
    if not found:
        raise KeyError(f"no {common_name!r} band in {sorted(item.assets)}")
    _, _, key, asset = min(found)
    return key, asset

In [ ]:
import os

import pandas as pd
import rioxarray  # registers .rio
import xarray as xr
from rasterio.enums import Resampling

# GDAL reads these at every (lazy) read; the S3 key is valid up to 8 h
os.environ.update(jstex.s3.gdal_env())
items = sorted(ex.results, key=lambda i: i.datetime)[:6]


def asset_by_prefix(item, prefix):
    found = [
        (a.extra_fields.get("gsd") or 999, k, a)
        for k, a in item.assets.items()
        if k.upper().startswith(prefix)
    ]
    return min(found)[1:] if found else (None, None)


def scaling(asset):
    """Scale, offset and nodata from the STAC raster metadata (asset level in STAC 1.1, else raster:bands)."""
    ef = asset.extra_fields
    rb = (ef.get("raster:bands") or [{}])[0]
    scale = ef.get("raster:scale", rb.get("scale", 1.0))
    offset = ef.get("raster:offset", rb.get("offset", 0.0))
    return scale, offset, ef.get("nodata", rb.get("nodata"))


def open_asset(asset):
    """Lazy, clipped to the AOI, scaled to physical values, nodata as NaN."""
    loc = jstex.s3.location(asset)
    da = rioxarray.open_rasterio(
        f"/vsis3/{loc['Bucket']}/{loc['Key']}", chunks={"x": 1024, "y": 1024}
    )
    da = da.squeeze("band", drop=True).rio.clip_box(*AOI, crs="EPSG:4326")
    scale, offset, nodata = scaling(asset)
    values = da * scale + offset
    return values if nodata is None else values.where(da != nodata)


frames, dates = [], []
template = None
for it in items:
    red = open_asset(band(it, "red")[1])
    nir = open_asset(band(it, "nir")[1])
    template = red if template is None else template
    red, nir = red.rio.reproject_match(template), nir.rio.reproject_match(template)
    ndvi = (nir - red) / (nir + red)
    _, scl_asset = asset_by_prefix(it, "SCL")
    if scl_asset is not None:  # keep vegetation, bare soil and water (SCL 4, 5, 6)
        scl = open_asset(scl_asset).rio.reproject_match(
            template, resampling=Resampling.nearest
        )
        ndvi = ndvi.where(scl.isin([4, 5, 6]))
    frames.append(ndvi)
    dates.append(pd.Timestamp(it.datetime))
cube = xr.concat(frames, dim=pd.Index(dates, name="time"))
cube

In [ ]:
series = cube.mean(dim=("x", "y")).compute()
series.plot(marker="o", figsize=(8, 3))

In [ ]:
cube.median("time").plot(cmap="RdYlGn", vmin=-1, vmax=1, figsize=(7, 6))

**Dask:** the arrays above are lazy (`chunks=`); `.compute()` reads only the clipped blocks. For larger areas or longer periods, start a `dask.distributed.Client()` first.